In [ ]:
!pip install selfies
!pip install tensorboard
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 30.2 MB/s eta 0:00:00


In [ ]:
!pip install selfies
!pip install tensorboard
!pip install rdkit

In [ ]:
data_path = "/content/drive/MyDrive/De novo Drug discovery"

def smiles_to_selfies(smiles):
    """Convert a SMILES string to SELFIES. Returns None for invalid SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Skip invalid molecules
    try:
        selfies_str = sf.encoder(smiles)
        return selfies_str
    except Exception as e:
        return None

# Load the CSV file with SMILES and property columns (e.g., QED, LogP, MolWt, etc.)
df = pd.read_csv(f"{data_path}/processed_molecules_2.csv")  # Adjust the filename as needed

# Convert SMILES to SELFIES
df['SELFIES'] = df['SMILES'].apply(smiles_to_selfies)

# Filter out rows where conversion failed
df = df[df['SELFIES'].notna()]

# Save the new CSV file with SMILES, SELFIES, and all the property columns
df.to_csv(f"{data_path}/zinc_selfies_with_props.csv", index=False)

print("Successfully generated SELFIES along with properties and saved to 'zinc_selfies_with_props.csv'.")


In [ ]:
#ADD SA-score component

#%% [code]
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import selfies as sf
from rdkit import Chem
from rdkit.Chem import QED, Crippen, Descriptors

#%% [code]
# === 1. Data Preparation: Load dataset and build vocabulary ===
# Assume you have a CSV with at least a 'SELFIES' column and pre-computed properties.
# For conditional training, your CSV should also have columns like 'QED', 'LogP', 'MolWt', etc.
data_path = "/content/drive/MyDrive/De novo Drug discovery/zinc_selfies_with_props.csv"
df = pd.read_csv(data_path)
print("Loaded", len(df), "molecules.")

# Tokenize SELFIES and build vocabulary
def tokenize_selfies(selfies_str):
    return list(sf.split_selfies(selfies_str))

all_tokens = []
for s in df['SELFIES']:
    tokens = tokenize_selfies(s)
    all_tokens.extend(tokens)

# Build vocabulary (add special tokens)
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab = special_tokens + sorted(set(all_tokens))
token2idx = {t: i for i, t in enumerate(vocab)}
idx2token = {i: t for t, i in token2idx.items()}
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

#%% [code]
# Create a PyTorch Dataset for conditional generation.
# Each sample: (SELFIES sequence as indices, property vector)
class SelfiesConditionalDataset(Dataset):
    def __init__(self, df, token2idx, max_len=100):
        self.df = df
        self.token2idx = token2idx
        self.max_len = max_len

        # Define the property columns you want to condition on
        self.prop_cols = ['QED', 'LogP', 'Molecular_Weight', 'Lipinski_H_Bond_Donors', 'Lipinski_H_Bond_Acceptors', 'Lipinski_Rule_Violation']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        selfies_str = row['SELFIES']
        tokens = ["<SOS>"] + tokenize_selfies(selfies_str) + ["<EOS>"]
        # Pad token sequence
        if len(tokens) < self.max_len:
            tokens += ["<PAD>"] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
        token_ids = [self.token2idx[t] for t in tokens]
        token_ids = torch.tensor(token_ids, dtype=torch.long)

        # Get property vector
        prop_vec = torch.tensor(row[self.prop_cols].astype(float).values, dtype=torch.float32)
        return token_ids, prop_vec

dataset = SelfiesConditionalDataset(df, token2idx, max_len=100)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4)
print("Dataset prepared, number of batches:", len(dataloader))

#%% [code]
# === 2. Model Definition ===
# We'll define a conditional Generator and a conditional Discriminator.
# The generator takes noise and a property vector as input, and outputs a sequence of logits over tokens.
# We use the Gumbel-Softmax trick to approximate sampling from a discrete distribution.

def gumbel_softmax_sample(logits, tau=1.0, eps=1e-10):
    U = torch.rand_like(logits)
    gumbel = -torch.log(-torch.log(U + eps) + eps)
    y = logits + gumbel
    return F.softmax(y / tau, dim=-1)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class ConditionalGenerator(nn.Module):
    def __init__(self, noise_dim, prop_dim, hidden_dim, vocab_size, max_len, embed_dim=256):
        super(ConditionalGenerator, self).__init__()
        self.noise_dim = noise_dim
        self.prop_dim = prop_dim
        self.hidden_dim = hidden_dim
        self.max_len = max_len
        self.vocab_size = vocab_size

        # Map noise and property vector to initial hidden state
        self.fc = nn.Linear(noise_dim + prop_dim, hidden_dim)

        # Embedding and LSTM decoder
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, noise, prop_vec, temperature=1.0):
        # noise: [batch, noise_dim]
        # prop_vec: [batch, prop_dim]
        batch_size = noise.size(0)
        # Concatenate noise and property vector, and map to hidden state
        h0 = torch.tanh(self.fc(torch.cat([noise, prop_vec], dim=1)))  # [batch, hidden_dim]
        h0 = h0.unsqueeze(0)  # [1, batch, hidden_dim]
        c0 = torch.zeros_like(h0)

        # Start token (<SOS>) for every sample
        sos_token = token2idx["<SOS>"]
        inputs = torch.full((batch_size, 1), sos_token, dtype=torch.long, device=noise.device)

        outputs = []
        hidden, cell = h0, c0
        for t in range(self.max_len - 1):
            emb = self.embedding(inputs)  # [batch, 1, embed_dim]
            out, (hidden, cell) = self.lstm(emb, (hidden, cell))  # out: [batch, 1, hidden_dim]
            logits = self.fc_out(out.squeeze(1))  # [batch, vocab_size]
            # Gumbel-Softmax sampling (differentiable approximation)
            probs = gumbel_softmax_sample(logits, tau=temperature, eps=1e-10)
            outputs.append(probs.unsqueeze(1))  # store for later loss calculation
            # For next time step, take the argmax (non-differentiable but used only in inference)
            inputs = torch.argmax(probs, dim=-1, keepdim=True)
        # Concatenate outputs along the sequence length dimension
        outputs = torch.cat(outputs, dim=1)  # [batch, max_len-1, vocab_size]
        return outputs

class ConditionalDiscriminator(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, prop_dim, max_len):
        super(ConditionalDiscriminator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim + prop_dim, 1)
        self.max_len = max_len

    def forward(self, token_ids, prop_vec):
        # token_ids: [batch, max_len] (discrete indices)
        emb = self.embedding(token_ids)  # [batch, max_len, embed_dim]
        _, (hidden, _) = self.lstm(emb)     # hidden: [1, batch, hidden_dim]
        hidden = hidden.squeeze(0)          # [batch, hidden_dim]
        x = torch.cat([hidden, prop_vec], dim=1)  # [batch, hidden_dim+prop_dim]
        out = self.fc(x)                    # [batch, 1]
        return out

    def forward_from_emb(self, emb, prop_vec):
        # emb: continuous embeddings, shape [batch, max_len, embed_dim]
        _, (hidden, _) = self.lstm(emb)     # hidden: [1, batch, hidden_dim]
        hidden = hidden.squeeze(0)          # [batch, hidden_dim]
        x = torch.cat([hidden, prop_vec], dim=1)  # [batch, hidden_dim+prop_dim]
        out = self.fc(x)                    # [batch, 1]
        return out

#%% [code]
# Hyperparameters
noise_dim = 100
prop_dim = 6   # number of conditioning properties
hidden_dim = 512
embed_dim = 256
max_len = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G = ConditionalGenerator(noise_dim, prop_dim, hidden_dim, vocab_size, max_len, embed_dim).to(device)
D = ConditionalDiscriminator(vocab_size, embed_dim, hidden_dim, prop_dim, max_len).to(device)

#%% [code]
# === 3. Losses and Optimizers ===
lr = 1e-4
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.0, 0.9))

# WGAN-GP hyperparameters
lambda_gp = 10

def gradient_penalty(D, real_samples, fake_samples, prop_vec):
    batch_size = real_samples.size(0)
    # Obtain embeddings from the discriminator's embedding layer
    real_emb = D.embedding(real_samples)   # shape: [batch, max_len, embed_dim]
    fake_emb = D.embedding(fake_samples)     # shape: [batch, max_len, embed_dim]

    # Create alpha with shape [batch, 1, 1] and expand to match real_emb
    alpha = torch.rand(batch_size, 1, 1, device=device)
    alpha = alpha.expand_as(real_emb)  # shape: [batch, max_len, embed_dim]

    # Interpolate in the embedding space
    interpolates = alpha * real_emb + (1 - alpha) * fake_emb
    interpolates.requires_grad_(True)

    # Disable cuDNN for the RNN forward pass in the gradient penalty calculation
    with torch.backends.cudnn.flags(enabled=False):
        d_interpolates = D.forward_from_emb(interpolates, prop_vec)

    fake = torch.ones(d_interpolates.size(), device=device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.reshape(batch_size, -1)
    grad_norm = gradients.norm(2, dim=1)
    gp = ((grad_norm - 1) ** 2).mean()
    return gp

#%% [code]
# === 4. Reinforcement Learning Reward Component ===
# Here we define a property reward function. For example, we can use QED.
# In practice you might combine several metrics.
def compute_property_reward(selfies_probs):
    """
    selfies_probs: [batch, max_len-1, vocab_size] output from G (Gumbel-softmax probabilities)
    To compute the reward, we need to decode the probabilities to SELFIES.
    Here, for simplicity, we take argmax at each time step.
    """
    batch_size = selfies_probs.size(0)
    # Get token indices from probabilities
    token_ids = torch.argmax(selfies_probs, dim=-1).cpu().numpy()  # shape [batch, max_len-1]
    rewards = []
    for seq in token_ids:
        # Convert indices to tokens (skip <SOS>)
        tokens = [idx2token[idx] for idx in seq if idx != token2idx["<PAD>"]]
        # Stop at <EOS>
        if "<EOS>" in tokens:
            tokens = tokens[:tokens.index("<EOS>")]
        selfies_str = "".join(tokens)
        try:
            smi = sf.decoder(selfies_str)
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                # For example, reward can be the QED score
                reward = QED.qed(mol)
            else:
                reward = 0.0
        except Exception as e:
            reward = 0.0
        rewards.append(reward)
    return torch.tensor(rewards, dtype=torch.float32, device=device)

#%% [code]
# === 5. Training Loop ===
num_epochs = 120  # Adjust based on your dataset and training needs
n_critic = 1      # Number of D updates per G update
writer = SummaryWriter(log_dir='/content/drive/MyDrive/De novo Drug discovery/training_logs')

for epoch in range(1, num_epochs+1):
    for i, (real_seq, prop_vec) in enumerate(dataloader):
        batch_size = real_seq.size(0)
        real_seq = real_seq.to(device)
        prop_vec = prop_vec.to(device)

        # Train Discriminator
        optimizer_D.zero_grad()

        # Sample noise and generate fake sequences from G
        noise = torch.randn(batch_size, noise_dim, device=device)
        fake_probs = G(noise, prop_vec, temperature=1.0)  # shape: [batch, max_len-1, vocab_size]

        # For D, we need discrete tokens.
        # Here we use argmax for real/fake samples (note: not differentiable, but only used for D)
        fake_seq = torch.argmax(fake_probs, dim=-1)  # [batch, max_len-1]
        # Pad fake_seq to max_len by adding a PAD token at the end if necessary.
        fake_seq = F.pad(fake_seq, (0, 1), value=token2idx["<PAD>"])

        # Discriminator outputs
        d_real = D(real_seq, prop_vec)
        d_fake = D(fake_seq.detach(), prop_vec)

        # Wasserstein loss and gradient penalty
        d_loss = -torch.mean(d_real) + torch.mean(d_fake)
        gp = gradient_penalty(D, real_seq, fake_seq.detach(), prop_vec)
        d_loss_total = d_loss + lambda_gp * gp

        d_loss_total.backward()
        optimizer_D.step()

        # Train Generator every n_critic iterations
        if i % n_critic == 0:
            optimizer_G.zero_grad()
            noise = torch.randn(batch_size, noise_dim, device=device)
            fake_probs = G(noise, prop_vec, temperature=1.0)
            fake_seq = torch.argmax(fake_probs, dim=-1)
            fake_seq = F.pad(fake_seq, (0, 1), value=token2idx["<PAD>"])

            # Standard generator loss (Wasserstein)
            g_loss_adv = -torch.mean(D(fake_seq, prop_vec))

            # Compute property reward from generated SELFIES
            prop_reward = compute_property_reward(fake_probs)
            prop_reward = (prop_reward - 0.5) * 2
            valid_count = (prop_reward > -0.9).sum().item()  # near 0 = invalid
            valid_percent = valid_count / batch_size * 100
            # We want to reward high QED, so we subtract the reward to reduce loss.
            reward_scale = 0.5  # can tune this over time
            g_loss = g_loss_adv - reward_scale * torch.mean(prop_reward)


            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
            optimizer_G.step()

    # Logging
    writer.add_scalar("Loss/Discriminator", d_loss_total.item(), epoch)
    writer.add_scalar("Loss/Generator", g_loss.item(), epoch)
    writer.add_scalar("PropertyReward", torch.mean(prop_reward).item(), epoch)
    writer.add_scalar("Reward/ValidPercent", valid_percent, epoch)
    print(f"Valid molecules: {valid_count}/{batch_size} ({valid_percent:.2f}%)")
    print(f"Epoch [{epoch}/{num_epochs}] d_loss: {d_loss_total.item():.4f} | g_loss: {g_loss.item():.4f} | Reward: {torch.mean(prop_reward).item():.4f}")

    # Optionally, save checkpoints every few epochs
    if epoch % 5 == 0:
        checkpoint_path = os.path.join("/content/drive/MyDrive/De novo Drug discovery/Checkpoints", f"checkpoint_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'G_state_dict': G.state_dict(),
            'D_state_dict': D.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict()
        }, checkpoint_path)
        print(f"Checkpoint saved at epoch {epoch}")

writer.close()
